# Neural Network Regression and Double-Descent Analysis

This notebook investigates neural-network regression for predicting continuous JNK3 and GSK3β docking scores using selected Mordred descriptors. A training-only mutual-information step reduces the descriptor set, after which one-hidden-layer neural networks of increasing capacity are trained using a fixed 50,000-step budget. Validation performance is compared across architectures to examine model-capacity behaviour and possible double descent. The validation-selected model is then evaluated once on the held-out scaffold test set, followed by SHAP-based interpretation of important descriptors.

In [ ]:
# Import all libraries required for this notebook

from pathlib import Path
from IPython.display import display

import copy
import gc
import json
import math
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.preprocessing import StandardScaler

from torch import nn
from torch.utils.data import (
    DataLoader,
    TensorDataset
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Computing device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

In [ ]:
# Load Mordred features, regression targets and seed 84 descriptors

random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

data_file = Path("model_dataset.csv")

training_feature_file = Path(
    "mordred_training_features_filtered.pkl"
)

validation_feature_file = Path(
    "mordred_validation_features_filtered.pkl"
)

seed_84_feature_file = Path(
    "full_ga_results_seed_84/"
    "full_ga_best_features.csv"
)

required_files = [
    data_file,
    training_feature_file,
    validation_feature_file,
    seed_84_feature_file
]

missing_files = [
    str(file)
    for file in required_files
    if not file.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

# Load the 626 filtered Mordred descriptors
X_train_all = pd.read_pickle(
    training_feature_file
).astype("float32")

X_valid_all = pd.read_pickle(
    validation_feature_file
).astype("float32")

# Load molecule information and continuous targets
model_data = pd.read_csv(data_file)

# Recover the original row index if it was saved
unnamed_columns = [
    column
    for column in model_data.columns
    if column.startswith("Unnamed")
]

if len(unnamed_columns) == 1:
    saved_index = unnamed_columns[0]

    if model_data[saved_index].is_unique:
        model_data = model_data.set_index(
            saved_index
        )
    else:
        model_data = model_data.drop(
            columns=unnamed_columns
        )

# Load the descriptors selected by GA seed 84
seed_84_features = pd.read_csv(
    seed_84_feature_file
)

seed_84_features = seed_84_features.loc[
    :,
    ~seed_84_features.columns.str.startswith(
        "Unnamed"
    )
]

if "descriptor" in seed_84_features.columns:
    feature_column = "descriptor"
else:
    feature_column = seed_84_features.columns[0]

ga_seed_84_descriptors = (
    seed_84_features[feature_column]
    .dropna()
    .astype(str)
    .tolist()
)

target_columns = [
    "jnk3_score",
    "gsk3b_score"
]

# Check the target columns
missing_targets = [
    target
    for target in target_columns
    if target not in model_data.columns
]

if missing_targets:
    raise ValueError(
        f"Missing target columns: {missing_targets}"
    )

# Check the seed 84 descriptors
missing_training_features = [
    feature
    for feature in ga_seed_84_descriptors
    if feature not in X_train_all.columns
]

missing_validation_features = [
    feature
    for feature in ga_seed_84_descriptors
    if feature not in X_valid_all.columns
]

if missing_training_features:
    raise ValueError(
        "Seed 84 descriptors missing from the "
        f"training data: {missing_training_features[:10]}"
    )

if missing_validation_features:
    raise ValueError(
        "Seed 84 descriptors missing from the "
        f"validation data: {missing_validation_features[:10]}"
    )

# Confirm that feature indices match the target dataset
if not X_train_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Training feature indices do not match "
        "model_dataset.csv."
    )

if not X_valid_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Validation feature indices do not match "
        "model_dataset.csv."
    )

# Keep the descriptors selected by seed 84
X_train_ga = X_train_all[
    ga_seed_84_descriptors
].copy()

X_valid_ga = X_valid_all[
    ga_seed_84_descriptors
].copy()

# Align continuous docking-score targets by row index
y_train_regression = model_data.loc[
    X_train_ga.index,
    target_columns
].astype("float32")

y_valid_regression = model_data.loc[
    X_valid_ga.index,
    target_columns
].astype("float32")

# Final checks
if y_train_regression.isna().any().any():
    raise ValueError(
        "Missing values found in training targets."
    )

if y_valid_regression.isna().any().any():
    raise ValueError(
        "Missing values found in validation targets."
    )

print(
    "Full training Mordred shape:",
    X_train_all.shape
)

print(
    "Full validation Mordred shape:",
    X_valid_all.shape
)

print(
    "\nSeed 84 selected descriptors:",
    len(ga_seed_84_descriptors)
)

print(
    "Seed 84 training feature shape:",
    X_train_ga.shape
)

print(
    "Seed 84 validation feature shape:",
    X_valid_ga.shape
)

print(
    "\nTraining target shape:",
    y_train_regression.shape
)

print(
    "Validation target shape:",
    y_valid_regression.shape
)

print(
    "Regression target order:",
    target_columns
)

print(
    "\nTraining and validation feature order matches:",
    X_train_ga.columns.equals(
        X_valid_ga.columns
    )
)

print(
    "Training features contain missing values:",
    X_train_ga.isna().any().any()
)

print(
    "Validation features contain missing values:",
    X_valid_ga.isna().any().any()
)

In [ ]:
# Reduce the GA descriptors to 100 using training data only

number_of_reduced_descriptors = 100

mi_results = pd.DataFrame({
    "descriptor": ga_seed_84_descriptors
})

for target in target_columns:

    mi_scores = mutual_info_regression(
        X_train_ga,
        y_train_regression[target],
        random_state=random_seed,
        n_jobs=-1
    )

    mi_results[
        f"{target}_mutual_information"
    ] = mi_scores

    mi_results[
        f"{target}_rank"
    ] = (
        mi_results[
            f"{target}_mutual_information"
        ]
        .rank(
            ascending=False,
            method="average"
        )
    )

# Average the ranks across both regression targets
mi_results["mean_rank"] = (
    mi_results[
        [
            "jnk3_score_rank",
            "gsk3b_score_rank"
        ]
    ]
    .mean(axis=1)
)

mi_results = (
    mi_results
    .sort_values(
        [
            "mean_rank",
            "descriptor"
        ]
    )
    .reset_index(drop=True)
)

reduced_descriptors = (
    mi_results
    .head(number_of_reduced_descriptors)[
        "descriptor"
    ]
    .tolist()
)

X_train_reduced = X_train_ga[
    reduced_descriptors
].copy()

X_valid_reduced = X_valid_ga[
    reduced_descriptors
].copy()

mi_results.to_csv(
    "seed84_descriptor_mutual_information_ranking.csv",
    index=False
)

pd.DataFrame({
    "descriptor": reduced_descriptors
}).to_csv(
    "seed84_top100_descriptors.csv",
    index=False
)

print(
    "Original seed 84 descriptors:",
    X_train_ga.shape[1]
)

print(
    "Reduced descriptors:",
    X_train_reduced.shape[1]
)

print(
    "\nReduced training feature shape:",
    X_train_reduced.shape
)

print(
    "Reduced validation feature shape:",
    X_valid_reduced.shape
)

display(
    mi_results.head(20)
)

print("\nSaved:")
print(
    "seed84_descriptor_mutual_information_ranking.csv"
)

print(
    "seed84_top100_descriptors.csv"
)

In [ ]:
# Standardise features and targets and prepare training batches

batch_size = 128
fixed_training_steps = 10_000

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

# Fit both scalers using training data only
X_train_scaled = feature_scaler.fit_transform(
    X_train_reduced
).astype("float32")

X_valid_scaled = feature_scaler.transform(
    X_valid_reduced
).astype("float32")

y_train_scaled = target_scaler.fit_transform(
    y_train_regression
).astype("float32")

y_valid_scaled = target_scaler.transform(
    y_valid_regression
).astype("float32")

# Convert arrays to PyTorch tensors
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_scaled,
    dtype=torch.float32
)

X_valid_tensor = torch.tensor(
    X_valid_scaled,
    dtype=torch.float32
)

y_valid_tensor = torch.tensor(
    y_valid_scaled,
    dtype=torch.float32
)

training_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

data_generator = torch.Generator()
data_generator.manual_seed(random_seed)

training_loader = DataLoader(
    training_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=True,
    generator=data_generator
)

steps_per_epoch = len(training_loader)

equivalent_epochs = (
    fixed_training_steps
    / steps_per_epoch
)

# Save the fitted scalers
joblib.dump(
    feature_scaler,
    "nn_top100_feature_scaler.pkl"
)

joblib.dump(
    target_scaler,
    "nn_regression_target_scaler.pkl"
)

print(
    "Standardised training feature shape:",
    X_train_tensor.shape
)

print(
    "Standardised validation feature shape:",
    X_valid_tensor.shape
)

print(
    "Standardised training target shape:",
    y_train_tensor.shape
)

print(
    "Standardised validation target shape:",
    y_valid_tensor.shape
)

print("\nBatch size:", batch_size)
print("Training batches per epoch:", steps_per_epoch)

print(
    "Fixed training steps:",
    fixed_training_steps
)

print(
    "Equivalent number of epochs:",
    round(equivalent_epochs, 2)
)

print("\nTraining feature mean:")
print(
    round(
        float(X_train_scaled.mean()),
        6
    )
)

print(
    "Training feature standard deviation:",
    round(
        float(X_train_scaled.std()),
        6
    )
)

print("\nSaved:")
print("nn_top100_feature_scaler.pkl")
print("nn_regression_target_scaler.pkl")

In [ ]:
# Define one-hidden-layer models and compare parameter counts

class OneHiddenLayerRegressor(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        output_size=2
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_size,
                hidden_size
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_size,
                output_size
            )
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train_tensor.shape[1]
output_size = y_train_tensor.shape[1]
number_of_training_samples = len(
    X_train_tensor
)

hidden_sizes = [
    10,
    50,
    100,
    250,
    500,
    1000,
    2000
]

parameter_results = []

for hidden_size in hidden_sizes:

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    )

    number_of_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    parameter_results.append({
        "hidden_neurons": hidden_size,
        "trainable_parameters":
            number_of_parameters,
        "training_samples":
            number_of_training_samples,
        "parameters_per_training_sample":
            number_of_parameters
            / number_of_training_samples,
        "capacity_regime": (
            "Modern"
            if number_of_parameters
            >= 2 * number_of_training_samples
            else "Classical/intermediate"
        )
    })

parameter_comparison = pd.DataFrame(
    parameter_results
)

parameter_comparison.to_csv(
    "nn_architecture_parameter_counts.csv",
    index=False
)

display(parameter_comparison)

print(
    "\nTraining samples:",
    number_of_training_samples
)

print(
    "Two times training samples:",
    2 * number_of_training_samples
)

print(
    "Three times training samples:",
    3 * number_of_training_samples
)

print(
    "\nSaved: nn_architecture_parameter_counts.csv"
)

In [ ]:
# Define fixed-budget neural-network training functions

results_directory = Path(
    "nn_double_descent_results"
)

results_directory.mkdir(
    exist_ok=True
)

learning_rate = 0.001
validation_interval = 100


def create_training_loader(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        training_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=True,
        generator=generator
    )


@torch.no_grad()
def calculate_validation_loss(
    model,
    validation_features,
    validation_targets,
    loss_function
):

    model.eval()

    predictions = model(
        validation_features
    )

    validation_loss = loss_function(
        predictions,
        validation_targets
    )

    return float(
        validation_loss.item()
    )


def train_fixed_budget_model(
    hidden_size,
    seed=random_seed
):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    training_loader_model = (
        create_training_loader(seed)
    )

    training_iterator = iter(
        training_loader_model
    )

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    ).to(device)

    number_of_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    validation_features_device = (
        X_valid_tensor.to(
            device,
            non_blocking=True
        )
    )

    validation_targets_device = (
        y_valid_tensor.to(
            device,
            non_blocking=True
        )
    )

    history_records = []
    interval_training_losses = []

    initial_validation_loss = (
        calculate_validation_loss(
            model,
            validation_features_device,
            validation_targets_device,
            loss_function
        )
    )

    history_records.append({
        "step": 0,
        "training_loss": np.nan,
        "validation_loss":
            initial_validation_loss
    })

    best_validation_loss = (
        initial_validation_loss
    )

    best_step = 0

    best_model_state = {
        name: value.detach().cpu().clone()
        for name, value
        in model.state_dict().items()
    }

    start_time = time.time()

    progress_bar = tqdm(
        range(
            1,
            fixed_training_steps + 1
        ),
        desc=(
            f"Hidden size {hidden_size}"
        )
    )

    for step in progress_bar:

        try:
            (
                batch_features,
                batch_targets
            ) = next(training_iterator)

        except StopIteration:
            training_iterator = iter(
                training_loader_model
            )

            (
                batch_features,
                batch_targets
            ) = next(training_iterator)

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_targets = batch_targets.to(
            device,
            non_blocking=True
        )

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        predictions = model(
            batch_features
        )

        training_loss = loss_function(
            predictions,
            batch_targets
        )

        training_loss.backward()
        optimizer.step()

        interval_training_losses.append(
            float(training_loss.item())
        )

        if (
            step % validation_interval == 0
            or step == fixed_training_steps
        ):

            mean_training_loss = float(
                np.mean(
                    interval_training_losses
                )
            )

            validation_loss = (
                calculate_validation_loss(
                    model,
                    validation_features_device,
                    validation_targets_device,
                    loss_function
                )
            )

            history_records.append({
                "step": step,
                "training_loss":
                    mean_training_loss,
                "validation_loss":
                    validation_loss
            })

            interval_training_losses = []

            if (
                validation_loss
                < best_validation_loss
            ):
                best_validation_loss = (
                    validation_loss
                )

                best_step = step

                best_model_state = {
                    name:
                        value.detach()
                        .cpu()
                        .clone()
                    for name, value
                    in model.state_dict().items()
                }

            progress_bar.set_postfix({
                "valid_loss":
                    f"{validation_loss:.5f}",
                "best_step":
                    best_step
            })

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    history = pd.DataFrame(
        history_records
    )

    model_file = (
        results_directory
        / f"hidden_{hidden_size}_best_model.pt"
    )

    history_file = (
        results_directory
        / f"hidden_{hidden_size}_history.csv"
    )

    torch.save(
        {
            "hidden_size": hidden_size,
            "input_size": input_size,
            "output_size": output_size,
            "trainable_parameters":
                number_of_parameters,
            "best_step": best_step,
            "best_validation_loss":
                best_validation_loss,
            "model_state_dict":
                best_model_state
        },
        model_file
    )

    history.to_csv(
        history_file,
        index=False
    )

    summary = {
        "hidden_neurons": hidden_size,
        "trainable_parameters":
            number_of_parameters,
        "best_step": best_step,
        "best_validation_loss":
            best_validation_loss,
        "final_validation_loss":
            float(
                history.iloc[-1][
                    "validation_loss"
                ]
            ),
        "elapsed_minutes":
            elapsed_minutes,
        "model_file":
            str(model_file),
        "history_file":
            str(history_file)
    }

    del model
    del optimizer
    del validation_features_device
    del validation_targets_device

    gc.collect()
    torch.cuda.empty_cache()

    return summary


print("Training function prepared.")
print(
    "Fixed training steps:",
    fixed_training_steps
)
print(
    "Validation interval:",
    validation_interval,
    "steps"
)
print("Learning rate:", learning_rate)
print("Batch size:", batch_size)
print(
    "Early stopping used:",
    False
)
print(
    "Dropout used:",
    False
)
print(
    "Results directory:",
    results_directory
)

In [ ]:
# Train all architectures for an extended fixed budget

fixed_training_steps = 50_000
validation_interval = 100

results_directory = Path(
    "nn_double_descent_results_50000"
)

results_directory.mkdir(
    exist_ok=True
)

extended_architecture_summaries = []

for hidden_size in hidden_sizes:

    model_file = (
        results_directory
        / f"hidden_{hidden_size}_best_model.pt"
    )

    history_file = (
        results_directory
        / f"hidden_{hidden_size}_history.csv"
    )

    # Load an existing completed run
    if model_file.exists() and history_file.exists():

        saved_history = pd.read_csv(
            history_file
        )

        if (
            int(saved_history["step"].max())
            == fixed_training_steps
        ):
            saved_model = torch.load(
                model_file,
                map_location="cpu"
            )

            extended_architecture_summaries.append({
                "hidden_neurons":
                    hidden_size,
                "trainable_parameters":
                    saved_model[
                        "trainable_parameters"
                    ],
                "best_step":
                    saved_model["best_step"],
                "best_validation_loss":
                    saved_model[
                        "best_validation_loss"
                    ],
                "final_validation_loss":
                    float(
                        saved_history.iloc[-1][
                            "validation_loss"
                        ]
                    ),
                "elapsed_minutes":
                    np.nan,
                "model_file":
                    str(model_file),
                "history_file":
                    str(history_file)
            })

            print(
                f"Hidden size {hidden_size}: "
                "completed 50,000-step run loaded."
            )

            continue

    print(
        f"\nTraining hidden size "
        f"{hidden_size} for "
        f"{fixed_training_steps:,} steps..."
    )

    model_summary = train_fixed_budget_model(
        hidden_size=hidden_size,
        seed=random_seed
    )

    extended_architecture_summaries.append(
        model_summary
    )

    # Save progress after every architecture
    pd.DataFrame(
        extended_architecture_summaries
    ).to_csv(
        results_directory
        / "nn_architecture_training_summary.csv",
        index=False
    )

extended_training_summary = (
    pd.DataFrame(
        extended_architecture_summaries
    )
    .sort_values(
        "hidden_neurons"
    )
    .reset_index(drop=True)
)

extended_training_summary[
    "parameters_per_training_sample"
] = (
    extended_training_summary[
        "trainable_parameters"
    ]
    / number_of_training_samples
)

extended_training_summary.to_csv(
    results_directory
    / "nn_architecture_training_summary.csv",
    index=False
)

display(
    extended_training_summary[
        [
            "hidden_neurons",
            "trainable_parameters",
            "parameters_per_training_sample",
            "best_step",
            "best_validation_loss",
            "final_validation_loss",
            "elapsed_minutes"
        ]
    ]
)

print(
    "\nCompleted architectures:",
    len(extended_training_summary)
)

print(
    "Fixed training budget:",
    f"{fixed_training_steps:,} steps"
)

print(
    "Saved:",
    results_directory
    / "nn_architecture_training_summary.csv"
)

In [ ]:
# Evaluate the best checkpoint from each architecture

@torch.no_grad()
def predict_in_batches(
    model,
    feature_tensor,
    prediction_batch_size=4096
):

    prediction_dataset = TensorDataset(
        feature_tensor
    )

    prediction_loader = DataLoader(
        prediction_dataset,
        batch_size=prediction_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    predictions = []

    model.eval()

    for (batch_features,) in prediction_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_predictions = model(
            batch_features
        )

        predictions.append(
            batch_predictions.cpu().numpy()
        )

    return np.concatenate(
        predictions,
        axis=0
    )


true_train_targets = (
    y_train_regression[
        target_columns
    ]
    .to_numpy()
)

true_valid_targets = (
    y_valid_regression[
        target_columns
    ]
    .to_numpy()
)

architecture_evaluation_results = []

for _, architecture_row in (
    extended_training_summary.iterrows()
):

    hidden_size = int(
        architecture_row[
            "hidden_neurons"
        ]
    )

    checkpoint_file = Path(
        architecture_row["model_file"]
    )

    checkpoint = torch.load(
        checkpoint_file,
        map_location="cpu"
    )

    model = OneHiddenLayerRegressor(
        input_size=input_size,
        hidden_size=hidden_size,
        output_size=output_size
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    train_predictions_scaled = (
        predict_in_batches(
            model,
            X_train_tensor
        )
    )

    valid_predictions_scaled = (
        predict_in_batches(
            model,
            X_valid_tensor
        )
    )

    train_predictions = (
        target_scaler.inverse_transform(
            train_predictions_scaled
        )
    )

    valid_predictions = (
        target_scaler.inverse_transform(
            valid_predictions_scaled
        )
    )

    result = {
        "hidden_neurons":
            hidden_size,
        "trainable_parameters":
            int(
                architecture_row[
                    "trainable_parameters"
                ]
            ),
        "parameters_per_training_sample":
            architecture_row[
                "parameters_per_training_sample"
            ],
        "best_step":
            int(
                architecture_row[
                    "best_step"
                ]
            ),
        "best_validation_loss":
            architecture_row[
                "best_validation_loss"
            ]
    }

    train_rmse_values = []
    valid_rmse_values = []
    train_r2_values = []
    valid_r2_values = []

    for target_index, target in enumerate(
        target_columns
    ):

        train_mae = mean_absolute_error(
            true_train_targets[
                :,
                target_index
            ],
            train_predictions[
                :,
                target_index
            ]
        )

        valid_mae = mean_absolute_error(
            true_valid_targets[
                :,
                target_index
            ],
            valid_predictions[
                :,
                target_index
            ]
        )

        train_rmse = np.sqrt(
            mean_squared_error(
                true_train_targets[
                    :,
                    target_index
                ],
                train_predictions[
                    :,
                    target_index
                ]
            )
        )

        valid_rmse = np.sqrt(
            mean_squared_error(
                true_valid_targets[
                    :,
                    target_index
                ],
                valid_predictions[
                    :,
                    target_index
                ]
            )
        )

        train_r2 = r2_score(
            true_train_targets[
                :,
                target_index
            ],
            train_predictions[
                :,
                target_index
            ]
        )

        valid_r2 = r2_score(
            true_valid_targets[
                :,
                target_index
            ],
            valid_predictions[
                :,
                target_index
            ]
        )

        result[
            f"{target}_train_mae"
        ] = train_mae

        result[
            f"{target}_valid_mae"
        ] = valid_mae

        result[
            f"{target}_train_rmse"
        ] = train_rmse

        result[
            f"{target}_valid_rmse"
        ] = valid_rmse

        result[
            f"{target}_train_r2"
        ] = train_r2

        result[
            f"{target}_valid_r2"
        ] = valid_r2

        train_rmse_values.append(
            train_rmse
        )

        valid_rmse_values.append(
            valid_rmse
        )

        train_r2_values.append(
            train_r2
        )

        valid_r2_values.append(
            valid_r2
        )

    result["mean_train_rmse"] = np.mean(
        train_rmse_values
    )

    result["mean_valid_rmse"] = np.mean(
        valid_rmse_values
    )

    result["rmse_generalisation_gap"] = (
        result["mean_valid_rmse"]
        - result["mean_train_rmse"]
    )

    result["mean_train_r2"] = np.mean(
        train_r2_values
    )

    result["mean_valid_r2"] = np.mean(
        valid_r2_values
    )

    architecture_evaluation_results.append(
        result
    )

    del model
    gc.collect()
    torch.cuda.empty_cache()


architecture_evaluation = (
    pd.DataFrame(
        architecture_evaluation_results
    )
    .sort_values(
        "hidden_neurons"
    )
    .reset_index(drop=True)
)

architecture_evaluation.to_csv(
    results_directory
    / "nn_architecture_validation_metrics.csv",
    index=False
)

selected_architecture = (
    architecture_evaluation.loc[
        architecture_evaluation[
            "best_validation_loss"
        ].idxmin()
    ]
)

display(
    architecture_evaluation[
        [
            "hidden_neurons",
            "trainable_parameters",
            "parameters_per_training_sample",
            "best_step",
            "best_validation_loss",
            "mean_train_rmse",
            "mean_valid_rmse",
            "rmse_generalisation_gap",
            "mean_valid_r2"
        ]
    ]
)

print(
    "\nValidation-selected hidden size:",
    int(
        selected_architecture[
            "hidden_neurons"
        ]
    )
)

print(
    "Selected trainable parameters:",
    int(
        selected_architecture[
            "trainable_parameters"
        ]
    )
)

print(
    "Selected best step:",
    int(
        selected_architecture[
            "best_step"
        ]
    )
)

print(
    "Selected minimum validation loss:",
    round(
        selected_architecture[
            "best_validation_loss"
        ],
        6
    )
)

print(
    "\nSaved:",
    results_directory
    / "nn_architecture_validation_metrics.csv"
)

In [ ]:
# Evaluate the validation-selected model on the test set

test_feature_file = Path(
    "mordred_test_features_filtered.pkl"
)

if not test_feature_file.exists():
    raise FileNotFoundError(
        f"Missing file: {test_feature_file}"
    )

# Load the untouched test descriptors
X_test_all = pd.read_pickle(
    test_feature_file
).astype("float32")

missing_test_features = [
    feature
    for feature in reduced_descriptors
    if feature not in X_test_all.columns
]

if missing_test_features:
    raise ValueError(
        "Selected descriptors missing from "
        f"the test set: {missing_test_features[:10]}"
    )

if not X_test_all.index.isin(
    model_data.index
).all():
    raise ValueError(
        "Test feature indices do not match "
        "model_dataset.csv."
    )

X_test_reduced = X_test_all[
    reduced_descriptors
].copy()

y_test_regression = model_data.loc[
    X_test_reduced.index,
    target_columns
].astype("float32")

if y_test_regression.isna().any().any():
    raise ValueError(
        "Missing values found in test targets."
    )

# Apply the training-set transformations
X_test_scaled = feature_scaler.transform(
    X_test_reduced
).astype("float32")

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

selected_hidden_size = int(
    selected_architecture[
        "hidden_neurons"
    ]
)

selected_checkpoint_file = Path(
    extended_training_summary.loc[
        extended_training_summary[
            "hidden_neurons"
        ] == selected_hidden_size,
        "model_file"
    ].iloc[0]
)

selected_checkpoint = torch.load(
    selected_checkpoint_file,
    map_location="cpu"
)

selected_model = OneHiddenLayerRegressor(
    input_size=input_size,
    hidden_size=selected_hidden_size,
    output_size=output_size
).to(device)

selected_model.load_state_dict(
    selected_checkpoint[
        "model_state_dict"
    ]
)

test_predictions_scaled = predict_in_batches(
    selected_model,
    X_test_tensor
)

test_predictions = target_scaler.inverse_transform(
    test_predictions_scaled
)

true_test_targets = (
    y_test_regression[
        target_columns
    ]
    .to_numpy()
)

final_metric_rows = []

for target_index, target in enumerate(
    target_columns
):

    # Previously calculated training metrics
    final_metric_rows.append({
        "split": "Train",
        "target": target,
        "mae": selected_architecture[
            f"{target}_train_mae"
        ],
        "rmse": selected_architecture[
            f"{target}_train_rmse"
        ],
        "r2": selected_architecture[
            f"{target}_train_r2"
        ]
    })

    # Previously calculated validation metrics
    final_metric_rows.append({
        "split": "Validation",
        "target": target,
        "mae": selected_architecture[
            f"{target}_valid_mae"
        ],
        "rmse": selected_architecture[
            f"{target}_valid_rmse"
        ],
        "r2": selected_architecture[
            f"{target}_valid_r2"
        ]
    })

    # Final untouched test metrics
    final_metric_rows.append({
        "split": "Test",
        "target": target,
        "mae": mean_absolute_error(
            true_test_targets[
                :,
                target_index
            ],
            test_predictions[
                :,
                target_index
            ]
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                true_test_targets[
                    :,
                    target_index
                ],
                test_predictions[
                    :,
                    target_index
                ]
            )
        ),
        "r2": r2_score(
            true_test_targets[
                :,
                target_index
            ],
            test_predictions[
                :,
                target_index
            ]
        )
    })

final_model_metrics = pd.DataFrame(
    final_metric_rows
)

test_prediction_data = pd.DataFrame(
    index=X_test_reduced.index
)

if "canonical_smiles" in model_data.columns:
    test_prediction_data[
        "canonical_smiles"
    ] = model_data.loc[
        X_test_reduced.index,
        "canonical_smiles"
    ]

for target_index, target in enumerate(
    target_columns
):
    test_prediction_data[
        f"actual_{target}"
    ] = true_test_targets[
        :,
        target_index
    ]

    test_prediction_data[
        f"predicted_{target}"
    ] = test_predictions[
        :,
        target_index
    ]

final_model_metrics.to_csv(
    results_directory
    / "selected_model_train_validation_test_metrics.csv",
    index=False
)

test_prediction_data.to_csv(
    results_directory
    / "selected_model_test_predictions.csv",
    index=False
)

display(final_model_metrics)

print(
    "\nSelected hidden neurons:",
    selected_hidden_size
)

print(
    "Selected trainable parameters:",
    selected_checkpoint[
        "trainable_parameters"
    ]
)

print(
    "Selected checkpoint step:",
    selected_checkpoint[
        "best_step"
    ]
)

print(
    "Test compounds:",
    len(X_test_reduced)
)

print("\nSaved:")
print(
    results_directory
    / "selected_model_train_validation_test_metrics.csv"
)
print(
    results_directory
    / "selected_model_test_predictions.csv"
)

In [ ]:
# Plot validation loss across neural-network capacities

capacity_plot_data = (
    extended_training_summary[
        [
            "hidden_neurons",
            "trainable_parameters",
            "best_validation_loss"
        ]
    ]
    .sort_values(
        "trainable_parameters"
    )
)

selected_parameters = int(
    selected_architecture[
        "trainable_parameters"
    ]
)

selected_loss = float(
    selected_architecture[
        "best_validation_loss"
    ]
)

plt.figure(
    figsize=(9, 6)
)

plt.plot(
    capacity_plot_data[
        "trainable_parameters"
    ],
    capacity_plot_data[
        "best_validation_loss"
    ],
    marker="o",
    linewidth=2
)

plt.scatter(
    selected_parameters,
    selected_loss,
    marker="*",
    s=220,
    label="Validation-selected model"
)

for _, row in capacity_plot_data.iterrows():

    plt.annotate(
        f'{int(row["hidden_neurons"])}',
        (
            row["trainable_parameters"],
            row["best_validation_loss"]
        ),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center"
    )

plt.axvline(
    number_of_training_samples,
    linestyle="--",
    linewidth=1.5,
    label="Number of training samples"
)

plt.axvline(
    2 * number_of_training_samples,
    linestyle=":",
    linewidth=1.5,
    label="Two times training samples"
)

plt.xscale("log")

plt.xlabel(
    "Number of trainable parameters "
    "(log scale)"
)

plt.ylabel(
    "Minimum validation MSE"
)

plt.title(
    "Neural-network capacity and "
    "validation performance"
)

plt.legend()
plt.grid(
    alpha=0.3
)

plt.tight_layout()

capacity_figure_file = (
    results_directory
    / "nn_capacity_validation_loss.png"
)

plt.savefig(
    capacity_figure_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    capacity_figure_file
)

In [ ]:
# Plot actual and predicted test docking scores, with best and worst predictions highlighted

from matplotlib.gridspec import GridSpec

target_display_names = {
    "jnk3_score": "JNK3",
    "gsk3b_score": "GSK3β"
}

target_colours = {
    "jnk3_score": "#3A8EC1",
    "gsk3b_score": "#B784E8"
}

panel_labels = {
    "jnk3_score": "A",
    "gsk3b_score": "B"
}

best_colour = "green"
worst_colour = "darkred"

prediction_figure_files = []

fig = plt.figure(figsize=(16, 9))

outer_gs = GridSpec(
    1, 2,
    figure=fig,
    wspace=0.55,
    left=0.08,
    right=0.95,
    top=0.85,
    bottom=0.15
)

legend_handles = []
legend_labels = []

for panel_index, target in enumerate(target_columns):

    actual_column = f"actual_{target}"
    predicted_column = f"predicted_{target}"

    actual_values = test_prediction_data[actual_column].to_numpy()
    predicted_values = test_prediction_data[predicted_column].to_numpy()

    r_squared = r2_score(actual_values, predicted_values)

    errors = np.abs(actual_values - predicted_values)
    best_idx = np.argsort(errors)[:3]
    worst_idx = np.argsort(errors)[-3:]

    scatter_colour = target_colours[target]

    # nested grid for this panel: histogram on top, scatter + right histogram below
    inner_gs = outer_gs[0, panel_index].subgridspec(
        4, 4, hspace=0.05, wspace=0.05
    )

    ax_main = fig.add_subplot(inner_gs[1:4, 0:3])
    ax_top = fig.add_subplot(inner_gs[0, 0:3], sharex=ax_main)
    ax_right = fig.add_subplot(inner_gs[1:4, 3], sharey=ax_main)

    # main scatter
    ax_main.scatter(
        actual_values,
        predicted_values,
        alpha=0.3,
        s=18,
        color=scatter_colour,
        edgecolor="none"
    )

    # ideal line
    lims = [
        min(actual_values.min(), predicted_values.min()) - 0.3,
        max(actual_values.max(), predicted_values.max()) + 0.3
    ]
    ideal_line, = ax_main.plot(lims, lims, linestyle="--", color="grey", linewidth=1)

    # R-squared annotation, top-left of the panel
    ax_main.text(
        0.05, 0.93,
        f"$R^2$ = {r_squared:.4f}",
        transform=ax_main.transAxes,
        fontsize=11,
        fontweight="bold",
        color=scatter_colour,
        va="top"
    )

    # best and worst predictions
    best_scatter = ax_main.scatter(
        actual_values[best_idx],
        predicted_values[best_idx],
        color=best_colour,
        s=80,
        zorder=5
    )

    worst_scatter = ax_main.scatter(
        actual_values[worst_idx],
        predicted_values[worst_idx],
        color=worst_colour,
        s=80,
        zorder=5
    )

    ax_main.set_xlim(lims)
    ax_main.set_ylim(lims)

    # best predictions labelled in a column outside the right edge of the axes
    best_label_ys = np.linspace(0.75, 0.25, len(best_idx))
    for i, y_frac in zip(best_idx, best_label_ys):
        ax_main.annotate(
            f"Index {i}",
            xy=(actual_values[i], predicted_values[i]),
            xycoords="data",
            xytext=(1.20, y_frac),
            textcoords="axes fraction",
            fontsize=9,
            color="black",
            ha="left",
            arrowprops=dict(arrowstyle="-", color=best_colour, lw=1.1),
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=0.7)
        )

    # worst predictions labelled in a column outside the left edge of the axes
    worst_label_ys = np.linspace(0.80, 0.20, len(worst_idx))
    for i, y_frac in zip(worst_idx, worst_label_ys):
        ax_main.annotate(
            f"Index {i}",
            xy=(actual_values[i], predicted_values[i]),
            xycoords="data",
            xytext=(-0.35, y_frac),
            textcoords="axes fraction",
            fontsize=9,
            color="black",
            ha="right",
            arrowprops=dict(arrowstyle="-", color=worst_colour, lw=1.1),
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=0.7)
        )

    ax_main.set_xlabel(f"Observed {target_display_names[target]} docking score")
    ax_main.set_ylabel(f"Predicted {target_display_names[target]} docking score")
    ax_main.grid(alpha=0.3)

    # panel letter and target name above each panel's histogram
    ax_top.set_title(
        f"{panel_labels[target]}   {target_display_names[target]}",
        loc="left",
        fontsize=12,
        fontweight="bold"
    )

    ax_top.hist(actual_values, bins=30, color=scatter_colour, alpha=0.6)
    ax_top.axis("off")

    ax_right.hist(predicted_values, bins=30, orientation="horizontal", color=scatter_colour, alpha=0.6)
    ax_right.axis("off")

    if panel_index == 0:
        legend_handles = [best_scatter, worst_scatter, ideal_line]
        legend_labels = ["Best predictions", "Worst predictions", "Ideal"]

# figure-level title
fig.suptitle(
    "Observed versus predicted docking scores on the scaffold test set",
    fontsize=15,
    y=0.95
)

# legend centered below the whole figure
fig.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.0)
)

output_file = "predictions_vs_true_docking_scores.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.show()

prediction_figure_files.append(output_file)

print("Saved:", prediction_figure_files)

In [ ]:
# SHAP analysis for the final neural network regressor

import shap

SHAP_SAMPLE_SIZE = 500
SHAP_RANDOM_SEED = 42

background_indices = np.random.RandomState(SHAP_RANDOM_SEED).choice(
    X_train_scaled.shape[0], size=100, replace=False
)
background_tensor = torch.tensor(
    X_train_scaled[background_indices], dtype=torch.float32
).to(device)

explain_sample_size = min(SHAP_SAMPLE_SIZE, X_test_scaled.shape[0])
explain_indices = np.random.RandomState(SHAP_RANDOM_SEED).choice(
    X_test_scaled.shape[0], size=explain_sample_size, replace=False
)
explain_tensor = torch.tensor(
    X_test_scaled[explain_indices], dtype=torch.float32
).to(device)

selected_model.eval()

explainer = shap.DeepExplainer(selected_model, background_tensor)
shap_values = explainer.shap_values(explain_tensor)

# check the shape returned before assuming a structure
print("Type of shap_values:", type(shap_values))

if isinstance(shap_values, list):
    print("Shape per output:", shap_values[0].shape)
else:
    print("Shape of combined array:", shap_values.shape)

for target_index, target in enumerate(target_columns):

    if isinstance(shap_values, list):
        # older shap versions: one array per output
        target_shap_values = shap_values[target_index]
    else:
        # newer shap versions: single array shaped (samples, features, outputs)
        target_shap_values = shap_values[:, :, target_index]

    mean_abs_shap = np.abs(target_shap_values).mean(axis=0)

    print(f"\n{target} - descriptors: {len(reduced_descriptors)}, shap values: {len(mean_abs_shap)}")

    shap_importance = pd.DataFrame({
        "descriptor": reduced_descriptors,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    print(f"\nTop 10 descriptors for {target}:")
    display(shap_importance.head(10))

    shap_importance.to_csv(
        results_directory / f"{target}_shap_importance.csv",
        index=False
    )

In [ ]:
# Plot best and worst 3 predictions by percentage error, annotated with the top SHAP descriptor

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import matplotlib.pyplot as plt
from mordred import Calculator, descriptors as mordred_descriptors

target = "jnk3_score"   # change to "gsk3b_score" for the other target

top_descriptor_by_target = {
    "jnk3_score": "AETA_eta_RL",
    "gsk3b_score": "ATS4Z"
}

top_descriptor = top_descriptor_by_target[target]

actual_column = f"actual_{target}"
predicted_column = f"predicted_{target}"

plot_df = test_prediction_data.copy()
plot_df["abs_error"] = (plot_df[actual_column] - plot_df[predicted_column]).abs()
plot_df["pct_error"] = (plot_df["abs_error"] / plot_df[actual_column].abs()) * 100

best_3 = plot_df.sort_values("pct_error").head(3)
worst_3 = plot_df.sort_values("pct_error", ascending=False).head(3)

# calculator to compute the specific top descriptor value for each molecule
calc = Calculator(mordred_descriptors, ignore_3D=True)

def get_descriptor_value(smiles, descriptor_name):
    mol = Chem.MolFromSmiles(smiles)
    result = calc(mol)

    # build a lookup dict from descriptor name to value, since Result
    # doesn't support .get() directly in this mordred version
    result_dict = {
        str(descriptor): value
        for descriptor, value in zip(calc.descriptors, result)
    }

    return result_dict.get(descriptor_name)

fig, axes = plt.subplots(2, 3, figsize=(15, 11))

row_groups = [
    (best_3, "Best 3 by percentage error"),
    (worst_3, "Worst 3 by percentage error")
]

for row_idx, (group_df, row_title) in enumerate(row_groups):

    for col_idx, (data_idx, row) in enumerate(group_df.iterrows()):

        ax = axes[row_idx, col_idx]

        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        mol_image = Draw.MolToImage(mol, size=(400, 300))

        ax.imshow(mol_image)
        ax.axis("off")

        ax.text(
            0.0, 1.08,
            f"Index {data_idx}",
            transform=ax.transAxes,
            fontsize=10,
            ha="left",
            va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.8)
        )

        true_value = row[actual_column]
        pred_value = row[predicted_column]
        abs_err = row["abs_error"]
        pct_err = row["pct_error"]

        mol_weight = Descriptors.MolWt(mol)
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        tpsa = Descriptors.TPSA(mol)
        logp = Descriptors.MolLogP(mol)

        descriptor_value = get_descriptor_value(row["canonical_smiles"], top_descriptor)

        stats_text = (
            f"True: {true_value:.3f}\n"
            f"Pred: {pred_value:.3f}\n"
            f"AbsErr: {abs_err:.3f} ({pct_err:.2f}%)\n"
            f"MW: {mol_weight:.1f} | RotB: {rot_bonds}\n"
            f"TPSA: {tpsa:.1f} | LogP: {logp:.2f}\n"
            f"{top_descriptor}: {descriptor_value:.3f}"
        )

        ax.text(
            0.5, -0.05,
            stats_text,
            transform=ax.transAxes,
            fontsize=9,
            ha="center",
            va="top",
            linespacing=1.8
        )

    fig.text(
        0.5,
        0.94 if row_idx == 0 else 0.46,
        row_title,
        fontsize=15,
        ha="center"
    )

plt.subplots_adjust(hspace=0.55, wspace=0.3, top=0.90, bottom=0.05)

output_file = f"best_worst_molecules_by_pct_error_{target}.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", output_file)

In [ ]:
# Check separation across best/worst molecules for all top-10 SHAP descriptors

from rdkit import Chem
from mordred import Calculator, descriptors as mordred_descriptors
import pandas as pd

target = "jnk3_score"   # change to "gsk3b_score" as needed

# paste in your actual top-10 list from the SHAP output for this target
top_10_descriptors = [
    "AETA_eta_RL",
    "ATS4Z",
    "AMID_C",
    "AATS4Z",
    "nRing",
    "RotRatio",
    "fragCpx",
    "AETA_beta",
    "AETA_beta_s",
    "ETA_dPsi_A"
]

actual_column = f"actual_{target}"
predicted_column = f"predicted_{target}"

plot_df = test_prediction_data.copy()
plot_df["abs_error"] = (plot_df[actual_column] - plot_df[predicted_column]).abs()
plot_df["pct_error"] = (plot_df["abs_error"] / plot_df[actual_column].abs()) * 100

best_3 = plot_df.sort_values("pct_error").head(3).copy()
worst_3 = plot_df.sort_values("pct_error", ascending=False).head(3).copy()

best_3["group"] = "Best"
worst_3["group"] = "Worst"

combined = pd.concat([best_3, worst_3])

calc = Calculator(mordred_descriptors, ignore_3D=True)

def get_all_descriptor_values(smiles, descriptor_names):
    mol = Chem.MolFromSmiles(smiles)
    result = calc(mol)
    result_dict = {
        str(descriptor): value
        for descriptor, value in zip(calc.descriptors, result)
    }
    return {name: result_dict.get(name) for name in descriptor_names}

descriptor_rows = []
for data_idx, row in combined.iterrows():
    values = get_all_descriptor_values(row["canonical_smiles"], top_10_descriptors)
    values["index"] = data_idx
    values["group"] = row["group"]
    values["pct_error"] = row["pct_error"]
    descriptor_rows.append(values)

descriptor_comparison = pd.DataFrame(descriptor_rows).set_index("index")

column_order = ["group", "pct_error"] + top_10_descriptors
descriptor_comparison = descriptor_comparison[column_order]

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print(f"Descriptor values across best/worst {target} predictions:\n")
display(descriptor_comparison)

print("\nMean values by group:")
display(descriptor_comparison.groupby("group")[top_10_descriptors].mean())

print("\nAbsolute difference between group means (larger = more separation):")
group_means = descriptor_comparison.groupby("group")[top_10_descriptors].mean()
separation = (group_means.loc["Best"] - group_means.loc["Worst"]).abs().sort_values(ascending=False)
display(separation)

descriptor_comparison.to_csv(f"{target}_descriptor_separation_check.csv")
print(f"\nSaved: {target}_descriptor_separation_check.csv")

In [ ]:
# Plot best and worst 3 predictions by percentage error, annotated with AMID_C

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import matplotlib.pyplot as plt
from mordred import Calculator, descriptors as mordred_descriptors

target = "jnk3_score"

top_descriptor = "AMID_C"

actual_column = f"actual_{target}"
predicted_column = f"predicted_{target}"

plot_df = test_prediction_data.copy()
plot_df["abs_error"] = (plot_df[actual_column] - plot_df[predicted_column]).abs()
plot_df["pct_error"] = (plot_df["abs_error"] / plot_df[actual_column].abs()) * 100

best_3 = plot_df.sort_values("pct_error").head(3)
worst_3 = plot_df.sort_values("pct_error", ascending=False).head(3)

calc = Calculator(mordred_descriptors, ignore_3D=True)

def get_descriptor_value(smiles, descriptor_name):
    mol = Chem.MolFromSmiles(smiles)
    result = calc(mol)
    result_dict = {
        str(descriptor): value
        for descriptor, value in zip(calc.descriptors, result)
    }
    return result_dict.get(descriptor_name)

fig, axes = plt.subplots(2, 3, figsize=(15, 11))

row_groups = [
    (best_3, "Best 3 by percentage error"),
    (worst_3, "Worst 3 by percentage error")
]

for row_idx, (group_df, row_title) in enumerate(row_groups):

    for col_idx, (data_idx, row) in enumerate(group_df.iterrows()):

        ax = axes[row_idx, col_idx]

        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        mol_image = Draw.MolToImage(mol, size=(400, 300))

        ax.imshow(mol_image)
        ax.axis("off")

        ax.text(
            0.0, 1.08,
            f"Index {data_idx}",
            transform=ax.transAxes,
            fontsize=10,
            ha="left",
            va="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.8)
        )

        true_value = row[actual_column]
        pred_value = row[predicted_column]
        abs_err = row["abs_error"]
        pct_err = row["pct_error"]

        mol_weight = Descriptors.MolWt(mol)
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        tpsa = Descriptors.TPSA(mol)
        logp = Descriptors.MolLogP(mol)

        descriptor_value = get_descriptor_value(row["canonical_smiles"], top_descriptor)

        stats_text = (
            f"True: {true_value:.3f}\n"
            f"Pred: {pred_value:.3f}\n"
            f"AbsErr: {abs_err:.3f} ({pct_err:.2f}%)\n"
            f"MW: {mol_weight:.1f} | RotB: {rot_bonds}\n"
            f"TPSA: {tpsa:.1f} | LogP: {logp:.2f}\n"
            f"{top_descriptor}: {descriptor_value:.3f}"
        )

        ax.text(
            0.5, -0.05,
            stats_text,
            transform=ax.transAxes,
            fontsize=9,
            ha="center",
            va="top",
            linespacing=1.8
        )

    fig.text(
        0.5,
        0.94 if row_idx == 0 else 0.46,
        row_title,
        fontsize=15,
        ha="center"
    )

plt.subplots_adjust(hspace=0.55, wspace=0.3, top=0.90, bottom=0.05)

output_file = f"best_worst_molecules_by_pct_error_{target}.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", output_file)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

In [ ]:
# Plot actual and predicted test docking scores, with best and worst predictions highlighted
from matplotlib.gridspec import GridSpec
target_display_names = {
    "jnk3_score": "JNK3",
    "gsk3b_score": "GSK3β"
}
target_colours = {
    "jnk3_score": "#3A8EC1",
    "gsk3b_score": "#B784E8"
}
panel_labels = {
    "jnk3_score": "A",
    "gsk3b_score": "B"
}
best_colour = "green"
worst_colour = "darkred"
prediction_figure_files = []
fig = plt.figure(figsize=(23, 13))
outer_gs = GridSpec(
    1, 2,
    figure=fig,
    wspace=0.8,
    left=0.08,
    right=0.95,
    top=0.85,
    bottom=0.15
)
legend_handles = []
legend_labels = []
for panel_index, target in enumerate(target_columns):
    actual_column = f"actual_{target}"
    predicted_column = f"predicted_{target}"
    actual_values = test_prediction_data[actual_column].to_numpy()
    predicted_values = test_prediction_data[predicted_column].to_numpy()
    r_squared = r2_score(actual_values, predicted_values)
    errors = np.abs(actual_values - predicted_values)
    best_idx = np.argsort(errors)[:3]
    worst_idx = np.argsort(errors)[-3:]
    scatter_colour = target_colours[target]
    # nested grid for this panel: histogram on top, scatter + right histogram below
    inner_gs = outer_gs[0, panel_index].subgridspec(
        4, 4, hspace=0.05, wspace=0.05
    )
    ax_main = fig.add_subplot(inner_gs[1:4, 0:3])
    ax_top = fig.add_subplot(inner_gs[0, 0:3], sharex=ax_main)
    ax_right = fig.add_subplot(inner_gs[1:4, 3], sharey=ax_main)
    # main scatter
    ax_main.scatter(
        actual_values,
        predicted_values,
        alpha=0.3,
        s=18,
        color=scatter_colour,
        edgecolor="none"
    )
    # ideal line
    lims = [
        min(actual_values.min(), predicted_values.min()) - 0.3,
        max(actual_values.max(), predicted_values.max()) + 0.3
    ]
    ideal_line, = ax_main.plot(lims, lims, linestyle="--", color="grey", linewidth=1)
    # R-squared annotation, top-left of the panel
    ax_main.text(
        0.05, 0.93,
        f"$R^2$ = {r_squared:.4f}",
        transform=ax_main.transAxes,
        fontsize=24,
        fontweight="bold",
        color=scatter_colour,
        va="top"
    )
    # best and worst predictions
    best_scatter = ax_main.scatter(
        actual_values[best_idx],
        predicted_values[best_idx],
        color=best_colour,
        s=100,
        zorder=5
    )
    worst_scatter = ax_main.scatter(
        actual_values[worst_idx],
        predicted_values[worst_idx],
        color=worst_colour,
        s=100,
        zorder=5
    )
    ax_main.set_xlim(lims)
    ax_main.set_ylim(lims)
    # best predictions labelled in a column outside the right edge of the axes
    best_label_ys = np.linspace(0.75, 0.25, len(best_idx))
    for i, y_frac in zip(best_idx, best_label_ys):
        ax_main.annotate(
            f"Index {i}",
            xy=(actual_values[i], predicted_values[i]),
            xycoords="data",
            xytext=(1.25, y_frac),
            textcoords="axes fraction",
            fontsize=21,
            color="black",
            ha="left",
            arrowprops=dict(arrowstyle="-", color=best_colour, lw=1.3),
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=0.9)
        )
    # worst predictions labelled in a column outside the left edge of the axes
    worst_label_ys = np.linspace(0.80, 0.20, len(worst_idx))
    for i, y_frac in zip(worst_idx, worst_label_ys):
        ax_main.annotate(
            f"Index {i}",
            xy=(actual_values[i], predicted_values[i]),
            xycoords="data",
            xytext=(-0.45, y_frac),
            textcoords="axes fraction",
            fontsize=21,
            color="black",
            ha="right",
            arrowprops=dict(arrowstyle="-", color=worst_colour, lw=1.3),
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", lw=0.9)
        )
    ax_main.set_xlabel(f"Observed {target_display_names[target]} docking score", fontsize=24, labelpad=14)
    ax_main.set_ylabel(f"Predicted {target_display_names[target]} docking score", fontsize=24, labelpad=14)
    ax_main.tick_params(axis='both', labelsize=19)
    ax_main.grid(alpha=0.3)
    # panel letter and target name above each panel's histogram
    ax_top.set_title(
        f"{panel_labels[target]}   {target_display_names[target]}",
        loc="left",
        fontsize=29,
        fontweight="bold"
    )
    ax_top.hist(actual_values, bins=30, color=scatter_colour, alpha=0.6)
    ax_top.axis("off")
    ax_right.hist(predicted_values, bins=30, orientation="horizontal", color=scatter_colour, alpha=0.6)
    ax_right.axis("off")
    if panel_index == 0:
        legend_handles = [best_scatter, worst_scatter, ideal_line]
        legend_labels = ["Best predictions", "Worst predictions", "Ideal"]
# figure-level title
fig.suptitle(
    "Observed versus predicted docking scores on the scaffold test set",
    fontsize=32,
    y=0.95
)
# legend centered below the whole figure
fig.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=23,
    bbox_to_anchor=(0.5, 0.0)
)
output_file = "predictions_vs_true_docking_scores.png"
plt.savefig(output_file, dpi=300, bbox_inches="tight")
plt.show()
prediction_figure_files.append(output_file)
print("Saved:", prediction_figure_files)

In [ ]:
# Plot best and worst 3 predictions by percentage error,
# with larger text for dissertation figure

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
import matplotlib.pyplot as plt
from mordred import Calculator, descriptors as mordred_descriptors

target = "jnk3_score"
top_descriptor = "AMID_C"

actual_column = f"actual_{target}"
predicted_column = f"predicted_{target}"

plot_df = test_prediction_data.copy()

plot_df["abs_error"] = (
    plot_df[actual_column] - plot_df[predicted_column]
).abs()

plot_df["pct_error"] = (
    plot_df["abs_error"] / plot_df[actual_column].abs()
) * 100

best_3 = plot_df.sort_values("pct_error").head(3)
worst_3 = plot_df.sort_values("pct_error", ascending=False).head(3)

calc = Calculator(mordred_descriptors, ignore_3D=True)


def get_descriptor_value(smiles, descriptor_name):
    mol = Chem.MolFromSmiles(smiles)

    result = calc(mol)

    result_dict = {
        str(descriptor): value
        for descriptor, value in zip(calc.descriptors, result)
    }

    return result_dict.get(descriptor_name)


# Slightly larger overall figure
fig, axes = plt.subplots(2, 3, figsize=(16, 12))

row_groups = [
    (best_3, "Best 3 by percentage error"),
    (worst_3, "Worst 3 by percentage error")
]


for row_idx, (group_df, row_title) in enumerate(row_groups):

    for col_idx, (data_idx, row) in enumerate(group_df.iterrows()):

        ax = axes[row_idx, col_idx]

        mol = Chem.MolFromSmiles(row["canonical_smiles"])

        # Keep molecular structure clear
        mol_image = Draw.MolToImage(
            mol,
            size=(500, 375)
        )

        ax.imshow(mol_image)
        ax.axis("off")

        # Larger index label
        ax.text(
            0.0,
            1.08,
            f"Index {data_idx}",
            transform=ax.transAxes,
            fontsize=14,
            ha="left",
            va="bottom",
            bbox=dict(
                boxstyle="round,pad=0.3",
                fc="white",
                ec="black",
                lw=1.0
            )
        )

        true_value = row[actual_column]
        pred_value = row[predicted_column]
        abs_err = row["abs_error"]
        pct_err = row["pct_error"]

        mol_weight = Descriptors.MolWt(mol)
        rot_bonds = Descriptors.NumRotatableBonds(mol)
        tpsa = Descriptors.TPSA(mol)
        logp = Descriptors.MolLogP(mol)

        descriptor_value = get_descriptor_value(
            row["canonical_smiles"],
            top_descriptor
        )

        stats_text = (
            f"True: {true_value:.3f}\n"
            f"Pred: {pred_value:.3f}\n"
            f"AbsErr: {abs_err:.3f} ({pct_err:.2f}%)\n"
            f"MW: {mol_weight:.1f} | RotB: {rot_bonds}\n"
            f"TPSA: {tpsa:.1f} | LogP: {logp:.2f}\n"
            f"{top_descriptor}: {descriptor_value:.3f}"
        )

        # Increased from 9 -> 12
        ax.text(
            0.5,
            -0.04,
            stats_text,
            transform=ax.transAxes,
            fontsize=12,
            ha="center",
            va="top",
            linespacing=1.5
        )

    # Larger row headings
    fig.text(
        0.5,
        0.95 if row_idx == 0 else 0.47,
        row_title,
        fontsize=19,
        fontweight="bold",
        ha="center"
    )


# Adjust spacing so larger text does not overlap
plt.subplots_adjust(
    hspace=0.62,
    wspace=0.24,
    top=0.90,
    bottom=0.06
)

output_file = f"best_worst_molecules_by_pct_error_{target}_large_text.png"

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", output_file)

In [ ]:
# FINAL corrected 10,000-epoch neural-network model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score
from matplotlib.gridspec import GridSpec

test_prediction_data = pd.read_csv(
    "nn_10000_epochs_results/selected_model_test_predictions.csv"
)

print("Loaded:", test_prediction_data.shape)
print(test_prediction_data.columns.tolist())

target_columns = [
    "jnk3_score",
    "gsk3b_score"
]

target_display_names = {
    "jnk3_score": "JNK3",
    "gsk3b_score": "GSK3β"
}

target_colours = {
    "jnk3_score": "#3A8EC1",
    "gsk3b_score": "#B784E8"
}

panel_labels = {
    "jnk3_score": "A",
    "gsk3b_score": "B"
}

best_colour = "green"
worst_colour = "darkred"

prediction_figure_files = []

fig = plt.figure(figsize=(23, 13))

outer_gs = GridSpec(
    1, 2,
    figure=fig,
    wspace=0.8,
    left=0.08,
    right=0.95,
    top=0.85,
    bottom=0.15
)

legend_handles = []
legend_labels = []

for panel_index, target in enumerate(target_columns):

    actual_column = f"actual_{target}"
    predicted_column = f"predicted_{target}"

    actual_values = (
        test_prediction_data[actual_column]
        .to_numpy()
    )

    predicted_values = (
        test_prediction_data[predicted_column]
        .to_numpy()
    )

    # Calculate R² from FINAL corrected predictions
    r_squared = r2_score(
        actual_values,
        predicted_values
    )

    print(
        f"{target_display_names[target]} R² = "
        f"{r_squared:.6f}"
    )

    # Absolute prediction error
    errors = np.abs(
        actual_values - predicted_values
    )

    # Three best and three worst predictions
    best_idx = np.argsort(errors)[:3]
    worst_idx = np.argsort(errors)[-3:]

    scatter_colour = target_colours[target]

    inner_gs = outer_gs[
        0, panel_index
    ].subgridspec(
        4, 4,
        hspace=0.05,
        wspace=0.05
    )

    ax_main = fig.add_subplot(
        inner_gs[1:4, 0:3]
    )

    ax_top = fig.add_subplot(
        inner_gs[0, 0:3],
        sharex=ax_main
    )

    ax_right = fig.add_subplot(
        inner_gs[1:4, 3],
        sharey=ax_main
    )

    ax_main.scatter(
        actual_values,
        predicted_values,
        alpha=0.3,
        s=18,
        color=scatter_colour,
        edgecolor="none"
    )

    lims = [
        min(
            actual_values.min(),
            predicted_values.min()
        ) - 0.3,

        max(
            actual_values.max(),
            predicted_values.max()
        ) + 0.3
    ]

    ideal_line, = ax_main.plot(
        lims,
        lims,
        linestyle="--",
        color="grey",
        linewidth=1
    )

    ax_main.text(
        0.05,
        0.93,
        f"$R^2$ = {r_squared:.4f}",
        transform=ax_main.transAxes,
        fontsize=24,
        fontweight="bold",
        color=scatter_colour,
        va="top"
    )

    best_scatter = ax_main.scatter(
        actual_values[best_idx],
        predicted_values[best_idx],
        color=best_colour,
        s=100,
        zorder=5
    )

    worst_scatter = ax_main.scatter(
        actual_values[worst_idx],
        predicted_values[worst_idx],
        color=worst_colour,
        s=100,
        zorder=5
    )


    ax_main.set_xlim(lims)
    ax_main.set_ylim(lims)

    best_label_ys = np.linspace(
        0.75,
        0.25,
        len(best_idx)
    )

    for i, y_frac in zip(
        best_idx,
        best_label_ys
    ):

        ax_main.annotate(
            f"Index {i}",
            xy=(
                actual_values[i],
                predicted_values[i]
            ),
            xycoords="data",
            xytext=(1.25, y_frac),
            textcoords="axes fraction",
            fontsize=21,
            color="black",
            ha="left",
            arrowprops=dict(
                arrowstyle="-",
                color=best_colour,
                lw=1.3
            ),
            bbox=dict(
                boxstyle="round,pad=0.35",
                fc="white",
                ec="black",
                lw=0.9
            )
        )

    worst_label_ys = np.linspace(
        0.80,
        0.20,
        len(worst_idx)
    )

    for i, y_frac in zip(
        worst_idx,
        worst_label_ys
    ):

        ax_main.annotate(
            f"Index {i}",
            xy=(
                actual_values[i],
                predicted_values[i]
            ),
            xycoords="data",
            xytext=(-0.45, y_frac),
            textcoords="axes fraction",
            fontsize=21,
            color="black",
            ha="right",
            arrowprops=dict(
                arrowstyle="-",
                color=worst_colour,
                lw=1.3
            ),
            bbox=dict(
                boxstyle="round,pad=0.35",
                fc="white",
                ec="black",
                lw=0.9
            )
        )

    ax_main.set_xlabel(
        f"Observed "
        f"{target_display_names[target]} "
        f"docking score",
        fontsize=24,
        labelpad=14
    )

    ax_main.set_ylabel(
        f"Predicted "
        f"{target_display_names[target]} "
        f"docking score",
        fontsize=24,
        labelpad=14
    )

    ax_main.tick_params(
        axis="both",
        labelsize=19
    )

    ax_main.grid(alpha=0.3)

    ax_top.set_title(
        f"{panel_labels[target]}   "
        f"{target_display_names[target]}",
        loc="left",
        fontsize=29,
        fontweight="bold"
    )

    ax_top.hist(
        actual_values,
        bins=30,
        color=scatter_colour,
        alpha=0.6
    )

    ax_top.axis("off")


    ax_right.hist(
        predicted_values,
        bins=30,
        orientation="horizontal",
        color=scatter_colour,
        alpha=0.6
    )

    ax_right.axis("off")


    if panel_index == 0:

        legend_handles = [
            best_scatter,
            worst_scatter,
            ideal_line
        ]

        legend_labels = [
            "Best predictions",
            "Worst predictions",
            "Ideal"
        ]

fig.suptitle(
    "Observed versus predicted docking scores "
    "on the scaffold test set",
    fontsize=32,
    y=0.95
)

fig.legend(
    handles=legend_handles,
    labels=legend_labels,
    loc="lower center",
    ncol=3,
    frameon=False,
    fontsize=23,
    bbox_to_anchor=(0.5, 0.0)
)

output_file = (
    "predictions_vs_true_docking_scores_FINAL.png"
)

plt.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

prediction_figure_files.append(
    output_file
)

print("\nSaved:", output_file)